In [ ]:
from pathlib import Path
from IPython.display import display
import json
import os
import re
import subprocess
import sys

WORKSPACE = Path('/kaggle/working')
PROJECT_DIR = WORKSPACE / 'point_cloud_playground_play'
REPO_URL = 'https://github.com/kslannmnd/point_cloud_playground_play.git'
GITHUB_TOKEN = ''
GITHUB_TOKEN_SECRET = 'GITHUB_TOKEN'
S3DIS_PREPROCESSED_INPUT = Path('/kaggle/input/datasets/ayukiseleva/s3dis-preprocessed')
S3DIS_PREPROCESSED_DIR = S3DIS_PREPROCESSED_INPUT
TEST_AREA = 'Area_5'
N_TEST_ROOMS = 3
N_TRAIN_ROOMS = 6
TRAIN_VARIANT = 'n_rooms'
METRIC_VARIANT = 'n_rooms'
PRETRAINED_CHECKPOINT = 'outputs/checkpoints/softgroup_s3dis_pretrained.pth'

EXPERIMENT_1 = {
    'training_config': 'colab_exp_1_pretrained_low_lr',
    'experiment_name': 'kaggle_exp_1_pretrained_low_lr',
    'label': 'kaggle_exp_1_pretrained_low_lr',
    'checkpoint': 'outputs/checkpoints/kaggle_exp_1_pretrained_low_lr_full_softgroup_latest.pth',
}
EXPERIMENT_2 = {
    'training_config': 'colab_exp_2_pretrained_medium_lr',
    'experiment_name': 'kaggle_exp_2_pretrained_medium_lr',
    'label': 'kaggle_exp_2_pretrained_medium_lr',
    'checkpoint': 'outputs/checkpoints/kaggle_exp_2_pretrained_medium_lr_full_softgroup_latest.pth',
}
EXPERIMENT_3 = {
    'training_config': 'colab_exp_3_two_stage_backbone_full',
    'experiment_name': 'kaggle_exp_3_two_stage_backbone_full',
    'label': 'kaggle_exp_3_two_stage_backbone_full',
    'checkpoint': 'outputs/checkpoints/kaggle_exp_3_two_stage_backbone_full_full_softgroup_latest.pth',
}
EXPERIMENTS = [EXPERIMENT_1, EXPERIMENT_2, EXPERIMENT_3]
RESULTS_BY_LABEL = {}

def run(cmd, cwd=None, env=None, check=True, display_cmd=None):
    text = display_cmd or (' '.join(map(str, cmd)) if isinstance(cmd, (list, tuple)) else str(cmd))
    print('\n[RUN]', text, '\n')
    proc = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        shell=isinstance(cmd, str),
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if proc.stdout:
        print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(f'Command failed with code {proc.returncode}:\n{text}')
    return proc

def get_github_token():
    token = GITHUB_TOKEN.strip()
    if token:
        return token
    token = os.environ.get('GITHUB_TOKEN')
    if token:
        return token.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET).strip()
    except Exception as exc:
        raise RuntimeError(f'Create Kaggle secret {GITHUB_TOKEN_SECRET!r} with private repo access first.') from exc

def authenticated_repo_url():
    token = get_github_token()
    if not token:
        raise RuntimeError('GitHub token is empty.')
    return REPO_URL.replace('https://', f'https://{token}@', 1)

assert S3DIS_PREPROCESSED_INPUT.exists(), S3DIS_PREPROCESSED_INPUT

## Clone and Install Project

In [ ]:
clone_url = authenticated_repo_url()
if PROJECT_DIR.exists():
    print('Project already exists:', PROJECT_DIR)
    run(['git', 'remote', 'set-url', 'origin', clone_url], cwd=PROJECT_DIR, display_cmd='git remote set-url origin https://***@github.com/kslannmnd/point_cloud_playground_play.git')
    run(['git', 'pull', '--ff-only'], cwd=PROJECT_DIR)
else:
    run(['git', 'clone', clone_url, str(PROJECT_DIR)], display_cmd=f'git clone https://***@github.com/kslannmnd/point_cloud_playground_play.git {PROJECT_DIR}')

run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=PROJECT_DIR, check=False, display_cmd='git remote set-url origin https://github.com/kslannmnd/point_cloud_playground_play.git')
run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip', 'wheel', 'setuptools'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=PROJECT_DIR)

## Find Preprocessed S3DIS and Discover Rooms

In [ ]:
def find_preprocessed_root(path):
    required = ['preprocess', 'preprocess_sample', 'val_gt']
    if all((path / name).is_dir() for name in required):
        return path
    for item in path.rglob('*') if path.exists() else []:
        if item.is_dir() and all((item / name).is_dir() for name in required):
            return item
    return None

s3dis_root = find_preprocessed_root(S3DIS_PREPROCESSED_INPUT)
assert s3dis_root is not None, S3DIS_PREPROCESSED_INPUT
S3DIS_PREPROCESSED_DIR = s3dis_root
print('S3DIS preprocessed root:', S3DIS_PREPROCESSED_DIR)

def natural_key(value):
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r'(\d+)', str(value))]

def discover_rooms(preprocess_dir):
    rooms = []
    for path in preprocess_dir.glob('*_inst_nostuff.pth'):
        match = re.match(r'^(Area_\d+)_(.+)_inst_nostuff\.pth$', path.name)
        if match:
            area, room = match.group(1), match.group(2)
            rooms.append({'area': area, 'room': room, 'scene': f'{area}_{room}'})
    return sorted(rooms, key=lambda item: (natural_key(item['area']), natural_key(item['room'])))

def room_ref(room):
    return f"{room['area']}/{room['room']}"

def list_override(name, rooms):
    return f"{name}=[{','.join(room_ref(room) for room in rooms)}]"

def metric_room_args(variant):
    if variant == 'all_test_rooms':
        return ['metrics.rooms=all_test_rooms']
    if variant == 'n_rooms':
        return [list_override('metrics.rooms', all_test_rooms[:N_TEST_ROOMS])]
    raise ValueError(variant)

def train_room_args(variant):
    if variant == 'all_trainable_rooms':
        return ['training.rooms=all_trainable_rooms']
    if variant == 'n_rooms':
        return [list_override('training.rooms', all_trainable_rooms[:N_TRAIN_ROOMS])]
    raise ValueError(variant)

all_rooms = discover_rooms(S3DIS_PREPROCESSED_DIR / 'preprocess')
all_test_rooms = [room for room in all_rooms if room['area'] == TEST_AREA]
all_trainable_rooms = [room for room in all_rooms if room['area'] != TEST_AREA]
assert all_test_rooms, TEST_AREA
assert all_trainable_rooms, TEST_AREA
print('all_test_rooms:', len(all_test_rooms))
print('all_trainable_rooms:', len(all_trainable_rooms))
print('metric rooms:', [room['scene'] for room in all_test_rooms[:N_TEST_ROOMS]] if METRIC_VARIANT == 'n_rooms' else 'all_test_rooms')
print('train rooms:', [room['scene'] for room in all_trainable_rooms[:N_TRAIN_ROOMS]] if TRAIN_VARIANT == 'n_rooms' else 'all_trainable_rooms')

## Kaggle SoftGroup Setup

In [ ]:
def gpu_arch_from_nvidia_smi():
    proc = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    name = (proc.stdout or '').strip().splitlines()[0] if proc.returncode == 0 and (proc.stdout or '').strip() else ''
    mapping = {'T4': '7.5', 'A100': '8.0', 'L4': '8.9', 'V100': '7.0', 'P100': '6.0'}
    for token, arch in mapping.items():
        if token in name:
            print('GPU:', name)
            return arch
    print('GPU:', name or 'unknown')
    return '7.5'

KAGGLE_CUDA_ARCH = gpu_arch_from_nvidia_smi()
COMMON = [
    'env=kaggle',
    'dataset=s3dis_preprocessed',
    'inference=s3dis',
    f'env.kaggle.input_s3dis_preprocessed_dir={S3DIS_PREPROCESSED_DIR}',
    f'dataset.preprocessed.source_dir={S3DIS_PREPROCESSED_DIR}',
    f'dataset.fold.preferred_test_area={TEST_AREA}',
    f'softgroup.runtime_env.CUMM_CUDA_ARCH_LIST={KAGGLE_CUDA_ARCH}',
    f'softgroup.runtime_env.TORCH_CUDA_ARCH_LIST={KAGGLE_CUDA_ARCH}',
    '++softgroup.runtime_env.MAX_JOBS=2',
]

run([sys.executable, 'scripts/prepare_kaggle_env.py', *COMMON], cwd=PROJECT_DIR)
run([sys.executable, 'scripts/install_dependencies_and_build_softgroup.py', *COMMON], cwd=PROJECT_DIR)
run([sys.executable, 'scripts/prepare_s3dis.py', *COMMON], cwd=PROJECT_DIR)
run([sys.executable, 'scripts/download_softgroup_checkpoints.py', *COMMON], cwd=PROJECT_DIR)
run([sys.executable, 'scripts/patch_softgroup_for_kaggle.py', *COMMON], cwd=PROJECT_DIR)
run([sys.executable, 'scripts/generate_softgroup_configs.py', *COMMON], cwd=PROJECT_DIR)

## Metrics Helpers

In [ ]:
def newest_run_dir(output_root):
    runs = sorted([path for path in output_root.iterdir() if path.is_dir()])
    assert runs, output_root
    return runs[-1]

def metric_output_dir(label):
    return PROJECT_DIR / 'outputs' / 'metrics' / 's3dis' / 'kaggle_experiments' / label

def read_metric_result(label, training_config, checkpoint):
    output_root = metric_output_dir(label)
    if not output_root.exists():
        return None
    run_dir = newest_run_dir(output_root)
    summary_path = run_dir / 'summary.json'
    if not summary_path.exists():
        return None
    row = {
        'label': label,
        'training_config': training_config,
        'checkpoint': checkpoint,
        'checkpoint_path': str(PROJECT_DIR / checkpoint),
        'metrics_dir': str(run_dir),
        'summary_path': str(summary_path),
    }
    row.update(json.loads(summary_path.read_text(encoding='utf-8')))
    return row

def run_metrics(label, checkpoint, training_config):
    output_dir = f'outputs/metrics/s3dis/kaggle_experiments/{label}'
    run([
        sys.executable, 'scripts/compute_s3dis_metrics.py', *COMMON,
        f'metrics.checkpoint={checkpoint}',
        f'metrics.output_dir={output_dir}',
        'metrics.force_run=true',
        'metrics.allow_precomputed_results=false',
        *metric_room_args(METRIC_VARIANT),
    ], cwd=PROJECT_DIR)
    result = read_metric_result(label, training_config, checkpoint)
    assert result is not None, metric_output_dir(label)
    RESULTS_BY_LABEL[label] = result
    return result

def run_experiment(exp):
    run([
        sys.executable, 'scripts/train.py', *COMMON,
        f"training={exp['training_config']}",
        f"training.experiment_name={exp['experiment_name']}",
        *train_room_args(TRAIN_VARIANT),
    ], cwd=PROJECT_DIR)
    checkpoint_path = PROJECT_DIR / exp['checkpoint']
    assert checkpoint_path.exists(), checkpoint_path
    result = run_metrics(exp['label'], exp['checkpoint'], exp['training_config'])
    result['expected_checkpoint'] = exp['checkpoint']
    result['experiment_name'] = exp['experiment_name']
    RESULTS_BY_LABEL[exp['label']] = result
    print('Checkpoint:', checkpoint_path)
    print('Metrics:', result['summary_path'])
    return result

## Baseline Metrics

In [ ]:
baseline_result = run_metrics('baseline_pretrained', PRETRAINED_CHECKPOINT, 'baseline')
print(json.dumps(baseline_result, indent=2))

## Experiment 1: Pretrained Low LR

In [ ]:
exp_1_result = run_experiment(EXPERIMENT_1)

## Experiment 2: Pretrained Medium LR

In [ ]:
exp_2_result = run_experiment(EXPERIMENT_2)

## Experiment 3: Two-Stage Backbone and Full SoftGroup

In [ ]:
exp_3_result = run_experiment(EXPERIMENT_3)

## Comparison Table

In [ ]:
import pandas as pd

comparison_items = [
    {'label': 'baseline_pretrained', 'training_config': 'baseline', 'checkpoint': PRETRAINED_CHECKPOINT},
    *EXPERIMENTS,
]
rows = []
for item in comparison_items:
    label = item['label']
    checkpoint = item['checkpoint']
    training_config = item['training_config']
    row = RESULTS_BY_LABEL.get(label) or read_metric_result(label, training_config, checkpoint)
    if row is not None:
        rows.append(row)

assert rows, 'No metrics found. Run baseline and at least one experiment cell first.'
baseline_row = next((row for row in rows if row['label'] == 'baseline_pretrained'), None)
baseline_score = None if baseline_row is None else baseline_row.get('weighted_correct_points_percent')
for row in rows:
    score = row.get('weighted_correct_points_percent')
    row['delta_weighted_correct_points_percent'] = None if score is None or baseline_score is None else score - baseline_score

columns = [
    'label',
    'training_config',
    'rooms',
    'objects',
    'weighted_correct_points_percent',
    'delta_weighted_correct_points_percent',
    'mean_room_correct_points_percent',
    'mean_room_matched_iou',
    'checkpoint_path',
    'summary_path',
]
comparison_df = pd.DataFrame(rows)
comparison_df = comparison_df[[column for column in columns if column in comparison_df.columns]]
comparison_root = PROJECT_DIR / 'outputs' / 'metrics' / 's3dis' / 'kaggle_experiments'
comparison_root.mkdir(parents=True, exist_ok=True)
comparison_csv = comparison_root / 'comparison_table.csv'
comparison_json = comparison_root / 'comparison_table.json'
comparison_df.to_csv(comparison_csv, index=False)
comparison_df.to_json(comparison_json, orient='records', indent=2)
display(comparison_df)
print('Comparison CSV:', comparison_csv)
print('Comparison JSON:', comparison_json)
print('Baseline checkpoint:', PROJECT_DIR / PRETRAINED_CHECKPOINT)
for exp in EXPERIMENTS:
    print(exp['experiment_name'], 'checkpoint:', PROJECT_DIR / exp['checkpoint'])
for row in rows:
    print(row['label'], 'metrics:', row['summary_path'])